In [12]:
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
from xgboost import XGBRegressor
import sklearn.ensemble
import pandas as pd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import seaborn as sns
import itertools 
import sklearn
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score

In [13]:
def calculate_elo_rating(df, initial_elo=1500, k=20):
    """
    Calcula o Elo Rating de cada equipa ao longo da época regular.
    Adiciona uma nova coluna 'elo_before_game' ao DataFrame com o valor de Elo antes de cada jogo.
    
    :param df: DataFrame com os jogos (incluindo GAME_ID, GAME_DATE, TEAM_ABBREVIATION, MATCHUP, WL)
    :param initial_elo: Elo inicial para todas as equipas
    :param k: fator de aprendizagem (K-factor)
    :return: DataFrame com a coluna 'elo_before_game' adicionada
    """
    import pandas as pd

    # Ordenar por data para garantir cronologia
    df = df.copy()
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
    df = df.sort_values(by='GAME_DATE')

    # Extrair equipas em casa e fora
    def get_home_away(row):
        if "vs." in row['MATCHUP']:
            return row['TEAM_ABBREVIATION'], row['MATCHUP'].split('vs. ')[1]
        elif "@" in row['MATCHUP']:
            return row['MATCHUP'].split('@ ')[1], row['TEAM_ABBREVIATION']
        return None, None

    df[['HOME_TEAM', 'AWAY_TEAM']] = df.apply(get_home_away, axis=1, result_type='expand')
    df = df.dropna(subset=['HOME_TEAM', 'AWAY_TEAM', 'WL'])

    # Inicializar ratings
    elo_ratings = {}
    elo_gamewise = []

    for game_id in df['GAME_ID'].unique():
        game_df = df[df['GAME_ID'] == game_id]

        if len(game_df) != 2:
            continue

        team1 = game_df.iloc[0]
        team2 = game_df.iloc[1]

        t1_tag = team1['TEAM_ABBREVIATION']
        t2_tag = team2['TEAM_ABBREVIATION']

        r1 = elo_ratings.get(t1_tag, initial_elo)
        r2 = elo_ratings.get(t2_tag, initial_elo)

        result1 = 1 if team1['WL'] == 'W' else 0
        result2 = 1 - result1

        elo_gamewise.append({
            'GAME_ID': game_id,
            f"{t1_tag}_elo_before": r1,
            f"{t2_tag}_elo_before": r2
        })

        # Função auxiliar
        def expected(ra, rb):
            return 1 / (1 + 10 ** ((rb - ra) / 400))

        e1 = expected(r1, r2)
        e2 = expected(r2, r1)

        elo_ratings[t1_tag] = r1 + k * (result1 - e1)
        elo_ratings[t2_tag] = r2 + k * (result2 - e2)

    # Criar DataFrame auxiliar
    elo_df = pd.DataFrame(elo_gamewise)

    # Função para recuperar o elo da equipa antes do jogo
    def get_elo(row):
        team = row['TEAM_ABBREVIATION']
        game_id = row['GAME_ID']
        match = elo_df[elo_df['GAME_ID'] == game_id]
        if not match.empty:
            return match.iloc[0].get(f"{team}_elo_before", None)
        return None

    df['elo_before_game'] = df.apply(get_elo, axis=1)
    return df


In [14]:
df = pd.read_csv("datasets/NBA_DATA_2010_2024/regular_season_totals_2010_2024.csv")

In [15]:
# Calcular o Elo Rating
df_com_elo = calculate_elo_rating(df)


In [17]:
display(df_com_elo[['TEAM_ABBREVIATION', 'GAME_ID', 'GAME_DATE', 'elo_before_game']])

,TEAM_ABBREVIATION,GAME_ID,GAME_DATE,elo_before_game
6649,LAL,21000003,2010-10-26,1500.000000
9213,BOS,21000001,2010-10-26,1500.000000
4394,POR,21000002,2010-10-26,1500.000000
9283,HOU,21000003,2010-10-26,1500.000000
6727,PHX,21000002,2010-10-26,1500.000000
...,...,...,...,...
9756,NOP,22301195,2024-04-14,1593.758594
32129,POR,22301200,2024-04-14,1322.435432
19548,NYK,22301190,2024-04-14,1597.258276
23458,HOU,22301199,2024-04-14,1491.607826


In [8]:
# Calucular Elo Rating last 10 games
df_com_elo = df_com_elo.sort_values(by=['TEAM_ABBREVIATION', 'GAME_DATE'])


df_com_elo['elo_last_10_avg'] = (
    df_com_elo
    .groupby('TEAM_ABBREVIATION')['elo_before_game']
    .transform(lambda x: x.shift(1).rolling(window=10, min_periods=1).mean())
)

display(df_com_elo[df_com_elo['TEAM_ABBREVIATION'] == 'BOS'][['GAME_ID','TEAM_ABBREVIATION','GAME_DATE', 'elo_before_game', 'elo_last_10_avg']].reset_index().head(10))

,index,GAME_ID,TEAM_ABBREVIATION,GAME_DATE,elo_before_game,elo_last_10_avg
0,9213,21000001,BOS,2010-10-26,1500.000000,NaN
1,29827,21000004,BOS,2010-10-27,1510.000000,1500.000000
2,22443,21000023,BOS,2010-10-29,1499.712256,1505.000000
3,12789,21000049,BOS,2010-11-02,1510.008275,1503.237419
4,26385,21000058,BOS,2010-11-03,1518.885850,1504.930133
5,26369,21000073,BOS,2010-11-05,1527.794377,1507.721276
6,26340,21000093,BOS,2010-11-07,1536.995800,1511.066793
7,26412,21000100,BOS,2010-11-08,1546.158939,1514.770937
8,6632,21000118,BOS,2010-11-11,1535.082438,1518.694437
9,12877,21000135,BOS,2010-11-13,1544.604698,1520.515326
